In [3]:
from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from azure.cognitiveservices.vision.computervision.models import OperationStatusCodes
from azure.cognitiveservices.vision.computervision.models import VisualFeatureTypes
from msrest.authentication import CognitiveServicesCredentials

from array import array
import os
from PIL import Image
import sys
import time

subscription_key = os.environ["VISION_KEY"]
endpoint = os.environ["VISION_ENDPOINT"]

computervision_client = ComputerVisionClient(endpoint, CognitiveServicesCredentials(subscription_key))

read_image_url = "https://www.cs.ubbcluj.ro/~sergiu/IA/test2.jpeg"

read_response = computervision_client.read(read_image_url,  raw=True)

read_operation_location = read_response.headers["Operation-Location"]

operation_id = read_operation_location.split("/")[-1]

while True:
    read_result = computervision_client.get_read_result(operation_id)
    if read_result.status not in ['notStarted', 'running']:
        break
    time.sleep(1)

ocr_boxes = []
ocr_texts = []

if read_result.status == OperationStatusCodes.succeeded:
    for text_result in read_result.analyze_result.read_results:
        for line in text_result.lines:
            print(line.text)
            ocr_texts.append(line.text)
            print(line.bounding_box)
            ocr_boxes.append(line.bounding_box)

Lucces in resolvarea
[86.0, 314.0, 1335.0, 287.0, 1336.0, 443.0, 86.0, 478.0]
TEMELOR la
[140.0, 590.0, 1045.0, 587.0, 1046.0, 723.0, 140.0, 727.0]
LABORA toarele de
[81.0, 915.0, 1007.0, 926.0, 1004.0, 1039.0, 78.0, 1014.0]
Inteligenta Artificialà!
[108.0, 1129.0, 1450.0, 1151.0, 1446.0, 1293.0, 105.0, 1259.0]


In [4]:
from scipy.spatial.distance import hamming

def levenshtein(s1, s2):
    n, m = len(s1), len(s2)
    if n == 0: return m
    if m == 0: return n
    D = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1):
        D[i][0] = i
    for j in range(m+1):
        D[0][j] = j
    for i in range(1,n+1):
        for j in range(1,m+1):
            cost = 0 if s1[i-1] == s2[j-1] else 1
            D[i][j] = min(D[i-1][j]+1, D[i][j-1]+1, D[i-1][j-1]+cost)
    return D[n][m]

def CER(gt, ocr):
    return levenshtein(gt, ocr) / max(len(gt),1)

def WER(gt, ocr):
    gt_words = gt.split()
    ocr_words = ocr.split()
    return levenshtein(gt_words, ocr_words) / max(len(gt_words),1)

def hamming_distance(gt, ocr):
    max_len = max(len(gt), len(ocr))
    gt_pad = gt.ljust(max_len)
    ocr_pad = ocr.ljust(max_len)
    return hamming(list(gt_pad), list(ocr_pad))

def jaro_winkler_approx(gt, ocr):
    return 1 - CER(gt, ocr)

def evaluate_line(gt, ocr):
    return {
        "Levenshtein": levenshtein(gt, ocr),
        "Hamming": round(hamming_distance(gt, ocr),3),
        "CER": round(CER(gt, ocr),3),
        "WER": round(WER(gt, ocr),3),
        "Jaro-Winkler-approx": round(jaro_winkler_approx(gt, ocr),3)
    }

gt_texts = [
    "Succes în rezolvarea",
    "tEMELOR la",
    "LABORAtoarele de",
    "Inteligență Artificială!"
]

for i, (gt, ocr) in enumerate(zip(gt_texts, ocr_texts)):
    res = evaluate_line(gt, ocr)
    print(f"Line {i+1} metrics:")
    for metric, val in res.items():
        print(f"  {metric}: {val}")
    print()

Line 1 metrics:
  Levenshtein: 3
  Hamming: 0.15
  CER: 0.15
  WER: 1.0
  Jaro-Winkler-approx: 0.85

Line 2 metrics:
  Levenshtein: 1
  Hamming: 0.1
  CER: 0.1
  WER: 0.5
  Jaro-Winkler-approx: 0.9

Line 3 metrics:
  Levenshtein: 1
  Hamming: 0.647
  CER: 0.062
  WER: 1.0
  Jaro-Winkler-approx: 0.938

Line 4 metrics:
  Levenshtein: 3
  Hamming: 0.125
  CER: 0.125
  WER: 1.0
  Jaro-Winkler-approx: 0.875



In [5]:
def box_to_coords(box):
    xs = box[0::2]
    ys = box[1::2]
    return min(xs), min(ys), max(xs), max(ys)

def iou(box1, box2):
    x1_min, y1_min, x1_max, y1_max = box_to_coords(box1)
    x2_min, y2_min, x2_max, y2_max = box_to_coords(box2)

    # Intersectie
    xi1 = max(x1_min, x2_min)
    yi1 = max(y1_min, y2_min)
    xi2 = min(x1_max, x2_max)
    yi2 = min(y1_max, y2_max)
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)

    # Reuniune
    box1_area = (x1_max - x1_min) * (y1_max - y1_min)
    box2_area = (x2_max - x2_min) * (y2_max - y2_min)
    union_area = box1_area + box2_area - inter_area

    return inter_area / union_area if union_area != 0 else 0

gt_boxes = [
    [80, 310, 1340, 480],
    [135, 585, 1050, 730],
    [80, 910, 1010, 1040],
    [105, 1125, 1455, 1300]
]

for i, (ocr_box, gt_box) in enumerate(zip(ocr_boxes, gt_boxes)):
    iou_val = iou(ocr_box, gt_box)
    correct = iou_val > 0.5
    print(f"Line {i+1}: IoU={iou_val:.2f}, Correctly localized={correct}")

num_correct = sum(iou(ocr_box, gt_box) > 0.5 for ocr_box, gt_box in zip(ocr_boxes, gt_boxes))
accuracy = num_correct / len(gt_boxes)
print(f"Localization accuracy: {accuracy*100:.2f}%")

Line 1: IoU=0.86, Correctly localized=True
Line 2: IoU=0.96, Correctly localized=True
Line 3: IoU=0.95, Correctly localized=True
Line 4: IoU=0.93, Correctly localized=True
Localization accuracy: 100.00%


Posibilitati de imbunatatire a recunoasterii textului

    -Imbunatatirea contrastului si luminozitatii pentru a evidentia textul

    -Corectarea orientarii (deskewing) si rotirea textului

    -Evitarea fotografiilor neclare sau miscate

    -Folosirea mai multor modele OCR sau combinarea rezultatelor pentru a creste acuratetea